In [ ]:
import pandas as pd

# Carrega a base integrada de migração e emprego
indicadores_integrados = pd.read_parquet(
    "../data/processed/indicadores_integrados_municipais.parquet"
)

# Carrega os indicadores municipais de educação técnica
educacao_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_educacao_municipal_2025.parquet"
)

# Integra as três dimensões pela chave municipal do IBGE
indicadores_completos = indicadores_integrados.merge(
    educacao_municipal.drop(columns=["municipio", "uf"], errors="ignore"),
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_educacao"
)

# Confere dimensões e cobertura entre as bases
print("Migração + emprego:", indicadores_integrados.shape)
print("Educação:", educacao_municipal.shape)
print("Base completa:", indicadores_completos.shape)

print("\nCobertura da integração:")
print(indicadores_completos["origem_integracao_educacao"].value_counts())

# Verifica se existem códigos IBGE duplicados
print("\nCódigos IBGE duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

In [ ]:
# Carrega a referência oficial dos municípios
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Adiciona os nomes e UFs de todos os municípios
indicadores_completos = indicadores_completos.drop(
    columns=["municipio", "uf"], errors="ignore"
).merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

# Verifica identificações geográficas ausentes
print("Municípios sem nome:", indicadores_completos["municipio"].isna().sum())
print("Municípios sem UF:", indicadores_completos["uf"].isna().sum())

# Verifica os valores ausentes em todas as colunas
display(
    indicadores_completos.isna().sum()
    .sort_values(ascending=False)
    .to_frame("valores_ausentes")
)

In [ ]:
# Identifica os municípios presentes nas fontes originais
indicadores_completos["tem_migracao"] = (
    indicadores_completos["2025"].notna()
)

indicadores_completos["tem_emprego"] = (
    indicadores_completos["admissoes"].notna()
)

indicadores_completos["tem_educacao"] = (
    indicadores_completos["ofertas_cursos"].notna()
)

# Conta quantas dimensões possuem dados em cada município
indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    ].sum(axis=1)
)

# Resume a cobertura das três dimensões
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Identifica quantos municípios possuem dados nas três dimensões
print(
    "Municípios com três dimensões:",
    (indicadores_completos["fontes_disponiveis"] == 3).sum()
)

In [ ]:
# Agrupa municípios pelas combinações de fontes disponíveis
cobertura_fontes = (
    indicadores_completos.groupby(
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    )
    .size()
    .reset_index(name="municipios")
)

# Ordena as combinações pelo número de municípios
cobertura_fontes = cobertura_fontes.sort_values(
    "municipios", ascending=False
)

# Exibe a distribuição
display(cobertura_fontes)

In [ ]:
# Seleciona municípios com registros de migração e emprego
municipios_priorizacao = indicadores_completos[
    indicadores_completos["tem_migracao"]
    & indicadores_completos["tem_emprego"]
].copy()

# Mantém informações territoriais e indicadores relevantes
colunas_priorizacao = [
    "codigo_ibge", "municipio", "uf",
    "2024", "2025", "variacao_absoluta", "variacao_%",
    "admissoes", "desligamentos", "saldo",
    "instituicoes_tecnicas", "cursos_distintos",
    "matriculas_tecnicas", "percentual_ead"
]

municipios_priorizacao = municipios_priorizacao[
    colunas_priorizacao
]

# Confere tamanho e cobertura educacional da base principal
print("Municípios elegíveis:", len(municipios_priorizacao))
print(
    "Com dados educacionais:",
    municipios_priorizacao["instituicoes_tecnicas"].notna().sum()
)

# Visualiza os primeiros registros
display(municipios_priorizacao.head())

In [ ]:
# Carrega os indicadores municipais do ensino superior
superior_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_ensino_superior_municipal_2024.parquet"
)

# Seleciona apenas a chave municipal e os indicadores educacionais
colunas_superior = [
    "codigo_ibge",
    "cursos_presenciais",
    "ies_presenciais",
    "matriculas_presenciais",
    "matriculas_estrangeiros_presencial",
    "matriculas_ead",
    "ingressantes_ead",
    "matriculas_estrangeiros_ead"
]

# Integra o ensino superior, preservando municípios de todas as fontes
indicadores_completos = indicadores_completos.merge(
    superior_municipal[colunas_superior],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_superior"
)

# Verifica a integração
print("Dimensões:", indicadores_completos.shape)
print("\nCobertura:")
print(indicadores_completos["origem_integracao_superior"].value_counts())

print("\nCódigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Municípios sem nome:",
      indicadores_completos["municipio"].isna().sum())

In [ ]:
# Carrega a referência oficial dos municípios

# Adiciona os nomes e UFs oficiais como colunas auxiliares
indicadores_completos = indicadores_completos.merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    suffixes=("", "_ibge"),
    validate="one_to_one"
)

# Preenche apenas os nomes e UFs que estavam ausentes
indicadores_completos["municipio"] = (
    indicadores_completos["municipio"]
    .fillna(indicadores_completos["municipio_ibge"])
)

indicadores_completos["uf"] = (
    indicadores_completos["uf"]
    .fillna(indicadores_completos["uf_ibge"])
)

# Remove as colunas auxiliares
indicadores_completos = indicadores_completos.drop(
    columns=["municipio_ibge", "uf_ibge"]
)

# Valida a identificação territorial
print("Municípios:", len(indicadores_completos))
print("Sem nome:", indicadores_completos["municipio"].isna().sum())
print("Sem UF:", indicadores_completos["uf"].isna().sum())
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

In [ ]:
# Atualiza os indicadores de presença das fontes
indicadores_completos["tem_migracao"] = indicadores_completos["2025"].notna()
indicadores_completos["tem_emprego"] = indicadores_completos["admissoes"].notna()
indicadores_completos["tem_educacao"] = indicadores_completos["ofertas_cursos"].notna()

# Identifica municípios com registros de ensino superior presencial ou EAD
indicadores_completos["tem_superior"] = indicadores_completos[
    ["matriculas_presenciais", "matriculas_ead"]
].notna().any(axis=1)

# Recalcula a quantidade de dimensões disponíveis
colunas_cobertura = [
    "tem_migracao",
    "tem_emprego",
    "tem_educacao",
    "tem_superior"
]

indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[colunas_cobertura].sum(axis=1)
)

# Exibe a distribuição de cobertura
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Confere a cobertura individual de cada fonte
display(
    indicadores_completos[colunas_cobertura]
    .sum()
    .to_frame("municipios")
)

In [ ]:
# Remove colunas auxiliares utilizadas durante as integrações
indicadores_completos = indicadores_completos.drop(
    columns=[
        "_merge",
        "origem_integracao_educacao",
        "origem_integracao_superior"
    ],
    errors="ignore"
)

# Confere os totais e a integridade da base
print("Municípios:", len(indicadores_completos))
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Registros migratórios 2025:",
      indicadores_completos["2025"].sum())

print("Admissões CAGED 2025:",
      indicadores_completos["admissoes"].sum())

print("Matrículas técnicas 2025:",
      indicadores_completos["matriculas_tecnicas"].sum())

print("Matrículas superiores presenciais 2024:",
      indicadores_completos["matriculas_presenciais"].sum())

print("Matrículas superiores EAD 2024:",
      indicadores_completos["matriculas_ead"].sum())

In [ ]:
# Define o caminho da base municipal consolidada
arquivo_saida = "../data/processed/indicadores_completos_municipais.parquet"

# Salva os indicadores integrados sem incluir o índice do DataFrame
indicadores_completos.to_parquet(
    arquivo_saida,
    index=False
)

# Confirma a exportação e suas dimensões
print("Arquivo salvo:", arquivo_saida)
print("Dimensões:", indicadores_completos.shape)

### Construindo o ranking territorial

In [ ]:
# Seleciona municípios presentes nas duas fontes principais
municipios_ranking = indicadores_completos[
    indicadores_completos["tem_migracao"]
    & indicadores_completos["tem_emprego"]
].copy()

# Examina a distribuição dos indicadores candidatos ao ranking
display(
    municipios_ranking[["2025", "admissoes"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

# Confere o número de municípios analisados
print("Municípios elegíveis:", len(municipios_ranking))

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Reduz a assimetria dos indicadores com transformação logarítmica
municipios_ranking["log_migracao"] = np.log1p(
    municipios_ranking["2025"]
)

municipios_ranking["log_admissoes"] = np.log1p(
    municipios_ranking["admissoes"]
)

# Normaliza os indicadores transformados para a escala de 0 a 1
scaler = MinMaxScaler()

municipios_ranking[["score_migracao", "score_emprego"]] = (
    scaler.fit_transform(
        municipios_ranking[["log_migracao", "log_admissoes"]]
    )
)

# Confere os intervalos dos indicadores normalizados
display(
    municipios_ranking[["score_migracao", "score_emprego"]]
    .describe()
)

In [ ]:
# Define cenários alternativos de ponderação
cenarios = {
    "equilibrado": (0.5, 0.5),
    "prioridade_migracao": (0.6, 0.4),
    "prioridade_emprego": (0.4, 0.6)
}

# Calcula a pontuação de cada município nos três cenários
for nome, (peso_migracao, peso_emprego) in cenarios.items():
    municipios_ranking[f"score_{nome}"] = (
        municipios_ranking["score_migracao"] * peso_migracao
        + municipios_ranking["score_emprego"] * peso_emprego
    )

# Calcula a posição de cada município em cada cenário
for nome in cenarios:
    municipios_ranking[f"rank_{nome}"] = (
        municipios_ranking[f"score_{nome}"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

# Exibe os 15 municípios mais bem posicionados no cenário equilibrado
display(
    municipios_ranking.sort_values("rank_equilibrado")[
        [
            "municipio", "uf", "2025", "admissoes",
            "score_equilibrado",
            "rank_equilibrado",
            "rank_prioridade_migracao",
            "rank_prioridade_emprego"
        ]
    ].head(15)
)

In [ ]:
# Calcula quanto a posição de cada município muda entre os cenários
municipios_ranking["variacao_ranking"] = (
    municipios_ranking["rank_prioridade_migracao"]
    - municipios_ranking["rank_prioridade_emprego"]
).abs()

# Resume a magnitude das mudanças de posição
print("Variação mediana:",
      municipios_ranking["variacao_ranking"].median())

print("Variação média:",
      municipios_ranking["variacao_ranking"].mean().round(2))

print("Maior variação:",
      municipios_ranking["variacao_ranking"].max())

# Mede a correlação entre as posições nos dois cenários extremos
correlacao = municipios_ranking[
    ["rank_prioridade_migracao", "rank_prioridade_emprego"]
].corr(method="spearman")

display(correlacao)

# Compara os municípios presentes no Top 20 dos três cenários
tops = {
    nome: set(municipios_ranking.nsmallest(
        20, f"rank_{nome}"
    )["codigo_ibge"])
    for nome in cenarios
}

print("Municípios comuns aos três Top 20:",
      len(set.intersection(*tops.values())))

### Investigar os municípios mais sensíveis
Antes de consolidar a metodologia, vamos identificar quais municípios apresentam as maiores diferenças de posição e entender por quê.

In [ ]:
# Seleciona os municípios com maiores mudanças entre os cenários
municipios_sensiveis = municipios_ranking.nlargest(
    15, "variacao_ranking"
)

# Compara os indicadores originais e as posições nos dois cenários
display(
    municipios_sensiveis[
        [
            "municipio", "uf",
            "2025", "admissoes",
            "rank_prioridade_migracao",
            "rank_prioridade_emprego",
            "variacao_ranking"
        ]
    ]
)

In [ ]:
# Cria uma cópia do ranking para a apresentação
ranking_municipal_2025 = municipios_ranking.copy()

# Converte a pontuação equilibrada para a escala de 0 a 100
ranking_municipal_2025["indice_oportunidade"] = (
    ranking_municipal_2025["score_equilibrado"] * 100
).round(2)

# Organiza os municípios pela posição no cenário equilibrado
ranking_municipal_2025 = ranking_municipal_2025.sort_values(
    "rank_equilibrado"
)

# Seleciona os indicadores essenciais para o ranking
colunas_ranking = [
    "codigo_ibge", "municipio", "uf",
    "2025", "admissoes", "saldo",
    "score_migracao", "score_emprego",
    "indice_oportunidade", "rank_equilibrado",
    "rank_prioridade_migracao", "rank_prioridade_emprego",
    "variacao_ranking"
]

ranking_municipal_2025 = ranking_municipal_2025[colunas_ranking]

# Exibe os dez primeiros municípios
display(ranking_municipal_2025.head(10))

In [ ]:
# Calcula a diferença entre os componentes normalizados do índice
ranking_municipal_2025["diferenca_componentes"] = (
    ranking_municipal_2025["score_migracao"]
    - ranking_municipal_2025["score_emprego"]
)

# Identifica municípios relativamente mais destacados em migração
print("Maior destaque migratório:")
display(
    ranking_municipal_2025.nlargest(10, "diferenca_componentes")[
        ["municipio", "uf", "2025", "admissoes", "diferenca_componentes"]
    ]
)

# Identifica municípios relativamente mais destacados em admissões
print("Maior destaque em emprego:")
display(
    ranking_municipal_2025.nsmallest(10, "diferenca_componentes")[
        ["municipio", "uf", "2025", "admissoes", "diferenca_componentes"]
    ]
)

## Carregando arquivos com dados sobre população geral Brasil (IBGE)

In [ ]:
# Define o caminho do arquivo Excel
arquivo_populacao = "../data/raw/populacao/POP2026_20261029.xlsx"

# Identifica as planilhas disponíveis
excel = pd.ExcelFile(arquivo_populacao)
print("Planilhas:", excel.sheet_names)

# Carrega a segunda planilha, com população municipal
populacao_municipal = pd.read_excel(
    arquivo_populacao,
    sheet_name=1,
    dtype={"COD. MUNIC": str}

)

# Remove colunas completamente vazias
populacao_municipal = populacao_municipal.dropna(
    axis=1,
    how="all"
).drop('Unnamed: 5', axis =1)



# Examina a estrutura e os primeiros registros
print("Dimensões:", populacao_municipal.shape)
print("Colunas:", populacao_municipal.columns.tolist())

display(populacao_municipal.head())

In [ ]:
# Concatena os códigos da UF e do município
populacao_municipal["codigo_ibge"] = (
    populacao_municipal["COD. UF"].astype(str).str.zfill(2)
    + populacao_municipal["COD. MUNIC"].astype(str).str.zfill(5)
).astype("int64")

# Renomeia a coluna de população para facilitar a integração
populacao_municipal = populacao_municipal.rename(
    columns={"POPULAÇÃO ESTIMADA": "populacao_2026"}
)

# Verifica a integridade dos dados
print("Municípios:", len(populacao_municipal))
print("Códigos duplicados:", populacao_municipal["codigo_ibge"].duplicated().sum())
print("Populações ausentes:", populacao_municipal["populacao_2026"].isna().sum())

# Confere a estrutura resultante
display(
    populacao_municipal[
        ["codigo_ibge", "NOME DO MUNICÍPIO", "UF", "populacao_2026"]
    ].head()
)

populacao_municipal[["codigo_ibge", "populacao_2026"]].to_parquet(
    "../data/processed/populacao_municipal_2026.parquet", index=False
)

### Participação de imigrantes nas admissões

In [ ]:
# Carrega os indicadores municipais do CAGED geral
emprego_geral = pd.read_parquet(
    "../data/processed/trabalho/emprego_geral_municipal_2025.parquet"
)

# Renomeia as variáveis para diferenciá-las dos indicadores de migrantes
emprego_geral = emprego_geral.rename(columns={
    "admissoes": "admissoes_gerais",
    "desligamentos": "desligamentos_gerais",
    "saldo": "saldo_geral"
})

# Integra o mercado geral à base consolidada, preservando todos os municípios
indicadores_completos = indicadores_completos.merge(
    emprego_geral[
        ["codigo_ibge", "admissoes_gerais", "desligamentos_gerais", "saldo_geral"]
    ],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_emprego_geral"
)

# Verifica as dimensões e a cobertura da integração
print("Dimensões:", indicadores_completos.shape)
print("Códigos duplicados:", indicadores_completos["codigo_ibge"].duplicated().sum())

display(indicadores_completos["origem_emprego_geral"].value_counts())

# Confere quantos municípios possuem os dois indicadores de admissões
print(
    "Municípios com dados de emprego geral e migrante:",
    (
        indicadores_completos["admissoes_gerais"].notna()
        & indicadores_completos["admissoes"].notna()
    ).sum()
)

In [ ]:
# Seleciona municípios com dados de emprego geral e migrante
comparacao_emprego = indicadores_completos[
    indicadores_completos["admissoes"].notna()
    & indicadores_completos["admissoes_gerais"].notna()
].copy()

# Calcula a diferença entre admissões gerais e de migrantes
comparacao_emprego["diferenca_admissoes"] = (
    comparacao_emprego["admissoes_gerais"]
    - comparacao_emprego["admissoes"]
)

# Identifica municípios com mais admissões migrantes do que gerais
inconsistencias = comparacao_emprego[
    comparacao_emprego["diferenca_admissoes"] < 0
]

print("Municípios comparados:", len(comparacao_emprego))
print("Municípios com inconsistências:", len(inconsistencias))

# Exibe eventuais inconsistências
display(
    inconsistencias[
        ["municipio", "uf", "admissoes", "admissoes_gerais",
         "diferenca_admissoes"]
    ]
)

In [ ]:
# Calcula a participação migrante apenas quando há admissões gerais
comparacao_emprego["participacao_migrante_pct"] = np.where(
    comparacao_emprego["admissoes_gerais"] > 0,
    comparacao_emprego["admissoes"]
    / comparacao_emprego["admissoes_gerais"] * 100,
    np.nan
)

# Verifica a distribuição e possíveis percentuais inválidos
print("Percentuais ausentes:",
      comparacao_emprego["participacao_migrante_pct"].isna().sum())

print("Percentuais acima de 100%:",
      (comparacao_emprego["participacao_migrante_pct"] > 100).sum())

display(
    comparacao_emprego["participacao_migrante_pct"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

# Examina os municípios com maior participação e pelo menos 100 admissões gerais
display(
    comparacao_emprego[
        comparacao_emprego["admissoes_gerais"] >= 100
    ]
    .nlargest(15, "participacao_migrante_pct")[
        ["municipio", "uf", "admissoes",
         "admissoes_gerais", "participacao_migrante_pct"]
    ]
)

In [ ]:
# Calcula os totais apenas nos 1.054 municípios comparáveis
total_migrantes = comparacao_emprego["admissoes"].sum()
total_geral = comparacao_emprego["admissoes_gerais"].sum()

# Calcula a participação ponderada pelo volume de admissões
participacao_ponderada = total_migrantes / total_geral * 100

print(f"Admissões migrantes: {total_migrantes:,.0f}")
print(f"Admissões gerais nos mesmos municípios: {total_geral:,.0f}")
print(f"Participação ponderada: {participacao_ponderada:.2f}%")
print(
    f"Média simples dos municípios: "
    f"{comparacao_emprego['participacao_migrante_pct'].mean():.2f}%"
)

**Interpretação**: os municípios com maior participação de migrantes nas admissões tendem a ter menor peso no volume total de contratações. Por isso, a média simples é superior à participação ponderada.

**Uma ressalva**: 1,77% é a participação nos 1.054 municípios comparados, não no Brasil inteiro. Além disso, precisamos verificar a compatibilidade dos recortes do CAGED antes de considerar essa participação definitiva.

In [ ]:
# Integra a população à base consolidada de municípios
indicadores_completos = indicadores_completos.merge(
    populacao_municipal[["codigo_ibge", "populacao_2026"]],
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Calcula registros migratórios por 10 mil habitantes
indicadores_completos["registros_por_10mil_hab"] = np.where(
    indicadores_completos["populacao_2026"] > 0,
    indicadores_completos["2025"]
    / indicadores_completos["populacao_2026"] * 10000,
    np.nan
)

# Verifica cobertura e distribuição do indicador
print("Municípios:", len(indicadores_completos))
print("Populações ausentes:",
      indicadores_completos["populacao_2026"].isna().sum())

display(
    indicadores_completos["registros_por_10mil_hab"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

Temos agora três perspectivas para analisar as oportunidades territoriais da UNIESF:
1. Volume: registros migratórios e admissões de migrantes.
2. Participação no emprego: percentual de admissões de migrantes no CAGED geral.
3. Intensidade populacional: registros migratórios por 10 mil habitantes.

In [ ]:
# Seleciona municípios com indicadores migratórios disponíveis
intensidade_migratoria = indicadores_completos[
    indicadores_completos["registros_por_10mil_hab"].notna()
].copy()

# Ordena os municípios pela intensidade dos registros migratórios
ranking_intensidade = intensidade_migratoria.sort_values(
    "registros_por_10mil_hab",
    ascending=False
)

# Exibe os 20 municípios com maior intensidade
display(
    ranking_intensidade[
        [
            "municipio",
            "uf",
            "2025",
            "populacao_2026",
            "registros_por_10mil_hab",
            "admissoes",
            "admissoes_gerais"
        ]
    ].head(20)
)

In [ ]:
# Seleciona os indicadores complementares da base consolidada
indicadores_relativos = indicadores_completos[
    ["codigo_ibge", "registros_por_10mil_hab",
     "admissoes", "admissoes_gerais"]
].copy()

# Integra os indicadores ao ranking de volume
comparacao_rankings = ranking_municipal_2025.merge(
    indicadores_relativos.drop(columns=["admissoes"]),
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Calcula a participação das admissões migrantes no emprego geral
comparacao_rankings["participacao_migrante_pct"] = (
    comparacao_rankings["admissoes"]
    / comparacao_rankings["admissoes_gerais"] * 100
)

# Verifica a distribuição dos três indicadores
display(
    comparacao_rankings[
        ["indice_oportunidade", "registros_por_10mil_hab",
         "participacao_migrante_pct"]
    ].describe()
)

# Examina as correlações entre as perspectivas
display(
    comparacao_rankings[
        ["indice_oportunidade", "registros_por_10mil_hab",
         "participacao_migrante_pct"]
    ].corr(method="spearman")
)

### Implicações para o projeto
Esses resultados sugerem dois perfis de oportunidades:

- Grandes polos de alcance: municípios com altos volumes absolutos de registros migratórios e contratações, como São Paulo e Curitiba.
- Territórios de maior intensidade: municípios onde os registros migratórios e as admissões de migrantes têm maior peso relativo, como Pacaraima e Chuí.

**Alguns municípios podem combinar os dois perfis.**

Para a UNIESF, essa distinção é útil porque municípios com grande volume podem representar maior alcance potencial, enquanto municípios de maior intensidade podem indicar contextos nos quais ações específicas para migrantes merecem investigação.

#### Normalizar os indicadores de intensidade
Assim como fizemos no ranking de volume, vamos usar log1p para reduzir a influência dos valores extremos e Min-Max para colocar as variáveis na mesma escala.

In [ ]:
# Cria uma cópia para construir o ranking de intensidade
ranking_intensidade_2025 = comparacao_rankings.copy()

# Aplica transformação logarítmica aos indicadores relativos
ranking_intensidade_2025["log_intensidade_migratoria"] = np.log1p(
    ranking_intensidade_2025["registros_por_10mil_hab"]
)

ranking_intensidade_2025["log_participacao_emprego"] = np.log1p(
    ranking_intensidade_2025["participacao_migrante_pct"]
)

# Normaliza os dois indicadores para a escala de 0 a 1
scaler_intensidade = MinMaxScaler()

ranking_intensidade_2025[
    ["score_intensidade_migratoria", "score_participacao_emprego"]
] = scaler_intensidade.fit_transform(
    ranking_intensidade_2025[
        ["log_intensidade_migratoria", "log_participacao_emprego"]
    ]
)

# Calcula o índice exploratório com pesos iguais
ranking_intensidade_2025["indice_intensidade"] = (
    0.5 * ranking_intensidade_2025["score_intensidade_migratoria"]
    + 0.5 * ranking_intensidade_2025["score_participacao_emprego"]
) * 100

# Exibe os 15 municípios com maior intensidade combinada
display(
    ranking_intensidade_2025.nlargest(15, "indice_intensidade")[
        [
            "municipio", "uf",
            "registros_por_10mil_hab",
            "participacao_migrante_pct",
            "indice_intensidade",
            "indice_oportunidade"
        ]
    ]
)

#### A análise permite distinguir três perfis preliminares:
- Polos de alcance: municípios com elevado volume absoluto, como São Paulo e Curitiba.
- Polos de intensidade: municípios com participação migrante expressiva em relação à população ou às contratações locais, como Pacaraima e Chuí.
- Polos mistos: municípios que se destacam nas duas perspectivas, como Boa Vista e Chapecó.

In [ ]:
# Calcula a posição dos municípios no ranking de intensidade
ranking_intensidade_2025["rank_intensidade"] = (
    ranking_intensidade_2025["indice_intensidade"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Compara a posição de volume e intensidade
ranking_intensidade_2025["diferenca_ranks"] = (
    ranking_intensidade_2025["rank_equilibrado"]
    - ranking_intensidade_2025["rank_intensidade"]
)

# Calcula a correlação de Spearman entre os rankings
correlacao = ranking_intensidade_2025[
    ["rank_equilibrado", "rank_intensidade"]
].corr(method="spearman").iloc[0, 1]

print(f"Correlação entre rankings: {correlacao:.3f}")

# Identifica municípios presentes nos dois Top 20
top_volume = set(
    ranking_intensidade_2025.nsmallest(20, "rank_equilibrado")["codigo_ibge"]
)

top_intensidade = set(
    ranking_intensidade_2025.nsmallest(20, "rank_intensidade")["codigo_ibge"]
)

print("Municípios comuns aos dois Top 20:",
      len(top_volume & top_intensidade))

# Exibe os municípios presentes nos dois grupos
display(
    ranking_intensidade_2025[
        ranking_intensidade_2025["codigo_ibge"].isin(top_volume & top_intensidade)
    ][
        ["municipio", "uf", "rank_equilibrado",
         "rank_intensidade", "indice_oportunidade",
         "indice_intensidade"]
    ].sort_values("rank_equilibrado")
)

In [ ]:
# Define cenários alternativos para os dois indicadores de intensidade
pesos_intensidade = {
    "equilibrado": (0.5, 0.5),
    "migracao": (0.6, 0.4),
    "emprego": (0.4, 0.6)
}

# Calcula os índices e rankings nos três cenários
for nome, (peso_migracao, peso_emprego) in pesos_intensidade.items():
    coluna_indice = f"indice_intensidade_{nome}"
    coluna_rank = f"rank_intensidade_{nome}"

    ranking_intensidade_2025[coluna_indice] = (
        ranking_intensidade_2025["score_intensidade_migratoria"] * peso_migracao
        + ranking_intensidade_2025["score_participacao_emprego"] * peso_emprego
    ) * 100

    ranking_intensidade_2025[coluna_rank] = (
        ranking_intensidade_2025[coluna_indice]
        .rank(ascending=False, method="min")
        .astype(int)
    )

# Compara a estabilidade entre os cenários extremos
correlacao_intensidade = ranking_intensidade_2025[
    ["rank_intensidade_migracao", "rank_intensidade_emprego"]
].corr(method="spearman").iloc[0, 1]

# Identifica municípios comuns aos três Top 20
tops_intensidade = [
    set(ranking_intensidade_2025.nsmallest(20, f"rank_intensidade_{nome}")["codigo_ibge"])
    for nome in pesos_intensidade
]

print(f"Correlação entre cenários: {correlacao_intensidade:.3f}")
print("Municípios comuns aos três Top 20:",
      len(set.intersection(*tops_intensidade)))

## Salvando os rankings

In [ ]:
# Recupera a população municipal da base integrada
ranking_intensidade_2025 = ranking_intensidade_2025.merge(
    indicadores_completos[["codigo_ibge", "populacao_2026"]],
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Confere se os 760 municípios possuem população
print("Municípios:", len(ranking_intensidade_2025))
print("Populações ausentes:",
      ranking_intensidade_2025["populacao_2026"].isna().sum())

In [ ]:
from pathlib import Path

# Define a pasta de saída dos rankings
pasta_saida = Path("../data/processed/rankings")
pasta_saida.mkdir(parents=True, exist_ok=True)

# Seleciona as colunas relevantes para análise e visualização
colunas_ranking = [
    "codigo_ibge", "municipio", "uf",
    "2025", "admissoes", "admissoes_gerais",
    "populacao_2026",
    "registros_por_10mil_hab",
    "participacao_migrante_pct",
    "indice_oportunidade", "rank_equilibrado",
    "indice_intensidade", "rank_intensidade",
    "rank_prioridade_migracao", "rank_prioridade_emprego",
    "rank_intensidade_migracao", "rank_intensidade_emprego"
]

# Organiza os resultados e salva o ranking comparativo
ranking_final_2025 = ranking_intensidade_2025[
    colunas_ranking
].sort_values("rank_equilibrado")

ranking_final_2025.to_parquet(
    pasta_saida / "ranking_municipal_volume_intensidade_2025.parquet",
    index=False
)

# Verifica o arquivo consolidado
print("Municípios:", len(ranking_final_2025))
print("Códigos duplicados:",
      ranking_final_2025["codigo_ibge"].duplicated().sum())

display(ranking_final_2025.head())